# Official FastCache Benchmark

This notebook executes the official FastCache pipeline on FLUX.1-schnell to evaluate performance and quality. It uses Kaggle-appropriate memory optimizations (4-bit quantization and CPU offloading) to run on Dual T4 GPUs.

In [ ]:
# Run this cell to clone the repository and install requirements on Kaggle
import os
if not os.path.exists('Sai-FastCache-xDiT'):
    !git clone https://github.com/Sainava/Sai-FastCache-xDiT.git
os.chdir('Sai-FastCache-xDiT')
!pip install torch numpy diffusers transformers accelerate sentencepiece protobuf bitsandbytes yunchang distvae einops

# Apply hotfix for FluxTransformerBlock config issue
fastcache_file = 'xfuser/model_executor/accelerator/fastcache.py'
with open(fastcache_file, 'r') as f:
    content = f.read()

old_code = "self.cache_projection = nn.Linear(model.config.hidden_size, model.config.hidden_size)"
new_code = """
        if hasattr(model, 'config') and hasattr(model.config, 'hidden_size'):
            hidden_size = model.config.hidden_size
        elif hasattr(model, 'hidden_size'):
            hidden_size = model.hidden_size
        elif hasattr(model, 'dim'):
            hidden_size = model.dim
        else:
            hidden_size = 3072
        self.cache_projection = nn.Linear(hidden_size, hidden_size)
"""

if old_code in content:
    content = content.replace(old_code, new_code)
    with open(fastcache_file, 'w') as f:
        f.write(content)


In [ ]:
import os
import sys
import time
import json
import csv
import torch
import numpy as np
from pathlib import Path
from diffusers import FluxPipeline
from diffusers.quantizers import PipelineQuantizationConfig

# Add current directory to path so we can import xfuser from the repo
sys.path.insert(0, str(Path.cwd()))

# Import official FastCache wrapper
from xfuser.model_executor.pipelines.fastcache_pipeline import xFuserFastCachePipelineWrapper

In [ ]:
# --- Configuration ---
MODEL_ID = "black-forest-labs/FLUX.1-schnell"

# Using official repository benchmark defaults
HEIGHT = 512
WIDTH = 512
NUM_STEPS = 10
CACHE_RATIO = 0.05
MOTION_THRESHOLD = 0.1
OUTPUT_DIR = "official_fastcache_images"

prompts = [
    "A highly detailed portrait of a cyberpunk hacker in a neon-lit room, cinematic lighting, intricate facial features, ultra realistic.",
    "A sweeping landscape of a distant alien planet with twin suns, dramatic mountains, reflective lakes, volumetric clouds, highly detailed.",
    "A massive brutalist concrete building standing alone in a misty forest, architectural photography, sharp geometric lines, overcast lighting.",
    "A close-up wildlife photograph of a magnificent snow leopard walking through fresh snow, ultra detailed fur, natural lighting, shallow depth of field.",
    "A magical fantasy castle floating on a giant rock above the clouds, waterfalls cascading into the sky, flying dragons, golden sunset, highly detailed concept art."
]
seeds = [42, 43, 44, 45, 46]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# --- 1. Load Model with Memory Optimization ---
print("Loading model with 4-bit quantization and CPU offloading...")
quant_config = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={
        "load_in_4bit": True,
        "bnb_4bit_compute_dtype": torch.bfloat16,
        "bnb_4bit_quant_type": "nf4"
    }
)

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_Token")
except ImportError:
    hf_token = os.environ.get("HF_TOKEN")
pipeline = FluxPipeline.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    token=hf_token
)
pipeline.enable_model_cpu_offload()

In [ ]:
# --- 2. Initialize Official FastCache ---
print("Initializing FastCache wrapper...")
fastcache_wrapper = xFuserFastCachePipelineWrapper(pipeline)
fastcache_wrapper.enable_fastcache(
    cache_ratio_threshold=CACHE_RATIO,
    motion_threshold=MOTION_THRESHOLD
)

In [ ]:
# --- 3. Execution ---
results = []
inference_times = []
total_start_time = time.perf_counter()

for i, (prompt, seed) in enumerate(zip(prompts, seeds)):
    print(f"\nRunning prompt {i+1}/{len(prompts)}: {prompt[:50]}...")
    
    # Generate seed
    generator = torch.Generator(device="cuda").manual_seed(seed)
    
    start_time = time.perf_counter()
    result = fastcache_wrapper(
        prompt=prompt,
        height=HEIGHT,
        width=WIDTH,
        num_inference_steps=NUM_STEPS,
        guidance_scale=0.0,
        generator=generator
    )
    end_time = time.perf_counter()
    
    inf_time = end_time - start_time
    inference_times.append(inf_time)
    
    # Save image
    image = result.images[0]
    img_path = os.path.join(OUTPUT_DIR, f"image_{i+1}_seed_{seed}.png")
    image.save(img_path)
    
    results.append({
        "Prompt": prompt,
        "Seed": seed,
        "Inference Time (seconds)": inf_time
    })
    
    print(f"Time: {inf_time:.2f}s | Saved to {img_path}")

total_end_time = time.perf_counter()
total_runtime = total_end_time - total_start_time

In [ ]:
# --- 4. Save Results and Config ---
csv_path = "official_fastcache_results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Prompt", "Seed", "Inference Time (seconds)"])
    writer.writeheader()
    writer.writerows(results)

config = {
    "model": MODEL_ID,
    "prompts": prompts,
    "seeds": seeds,
    "image_size": {"height": HEIGHT, "width": WIDTH},
    "inference_steps": NUM_STEPS,
    "scheduler": pipeline.scheduler.__class__.__name__,
    "FastCache_configuration": {
        "cache_ratio_threshold": CACHE_RATIO,
        "motion_threshold": MOTION_THRESHOLD
    },
    "quantization_settings": {
        "load_in_4bit": True,
        "compute_dtype": "bfloat16",
        "quant_type": "nf4"
    },
    "memory_settings": {
        "cpu_offload": True
    }
}

json_path = "official_fastcache_config.json"
with open(json_path, "w") as f:
    json.dump(config, f, indent=4)

In [ ]:
# --- 5. Print Summary ---
avg_time = np.mean(inference_times)
std_time = np.std(inference_times)

print("\n" + "="*40)
print("EXPERIMENT SUMMARY")
print("="*40)
print(f"Average Inference Time: {avg_time:.2f} seconds")
print(f"Standard Deviation:     {std_time:.2f} seconds")
print(f"Total Script Runtime:   {total_runtime:.2f} seconds")
print("="*40)